# Μάθημα 10 - Πράκτορες Τεχνητής Νοημοσύνης στην Παραγωγή

Σε αυτό το μάθημα θα μάθετε **παραδείγματα παραγωγής** για πράκτορες τεχνητής νοημοσύνης χρησιμοποιώντας το Microsoft Agent Framework (Python). Καλύπτουμε:

- **Παρατηρησιμότητα** — προσθήκη χρονομέτρησης και καταγραφής στις αλληλεπιδράσεις του πράκτορα
- **Αξιολόγηση** — χρήση κριτήρια πράκτορα για βαθμολόγηση της ποιότητας των απαντήσεων
- **Διαχείριση κόστους** — στρατηγικές για βελτιστοποίηση συμβόλων και επιλογή μοντέλου

Το σενάριο είναι ένας **ταξιδιωτικός πράκτορας** που βοηθά τους χρήστες να σχεδιάζουν ταξίδια, με παρακολούθηση και αξιολόγηση να εφαρμόζονται επιπλέον.


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Παράγοντες Παραγωγής

Η μεταφορά των πρακτόρων AI από πρωτότυπα στην παραγωγή απαιτεί προσεκτική προσοχή σε τρεις πυλώνες:

1. **Παρατηρησιμότητα** — Χρειάζεστε ορατότητα στο τι κάνει ο πράκτορας, πόσο χρόνο παίρνει και ποια εργαλεία καλεί. Χωρίς παρακολούθηση και καταγραφή, η αποσφαλμάτωση προβλημάτων στην παραγωγή είναι σχεδόν αδύνατη.

2. **Αξιολόγηση** — Αυτόματοι έλεγχοι ποιότητας διασφαλίζουν ότι οι απαντήσεις του πράκτορα παραμένουν ακριβείς, ολοκληρωμένες και χρήσιμες με την πάροδο του χρόνου. Ένας πράκτορας αξιολογητής μπορεί να βαθμολογεί τις απαντήσεις βάσει καθορισμένων κριτηρίων.

3. **Διαχείριση Κόστους** — Η χρήση token επηρεάζει άμεσα το κόστος. Στρατηγικές όπως η βελτιστοποίηση των prompts, η επιλογή μοντέλου και η αποθήκευση στην προσωρινή μνήμη βοηθούν στον έλεγχο των εξόδων χωρίς να θυσιάζεται η ποιότητα.


## Δημιουργία ενός Παρατηρήσιμου Πράκτορα

Καθορίζουμε τα εργαλεία ταξιδιού και περικλείουμε την κλήση του πράκτορα με μέτρηση χρόνου ώστε να μπορούμε να παρακολουθούμε την καθυστέρηση. Στην παραγωγή θα ενσωματώνατε το OpenTelemetry ή ένα παρόμοιο σύστημα εντοπισμού.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Πρότυπα Αξιολόγησης

Ένα κοινό πρότυπο παραγωγής είναι να χρησιμοποιείται ένας δεύτερος πράκτορας ως **αξιολογητής**. Ο αξιολογητής βαθμολογεί την απάντηση του πρωτευόντος πράκτορα βάσει προκαθορισμένων κριτηρίων όπως πληρότητα, ακρίβεια και χρηστικότητα.

Αυτό επιτρέπει:
- Αυτοματοποιημένες πύλες ποιότητας πριν οι απαντήσεις φτάσουν στους χρήστες
- Ανίχνευση παλινδρόμησης όταν αλλάζουν τα προτροπές ή τα μοντέλα
- Συνεχή παρακολούθηση της απόδοσης του πράκτορα με την πάροδο του χρόνου


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Στρατηγικές Διαχείρισης Κόστους

Ο έλεγχος των εξόδων είναι κρίσιμος για παραγωγικούς πράκτορες AI. Εδώ είναι βασικές στρατηγικές:

| Στρατηγική | Περιγραφή |
|---|---|
| **Βελτιστοποίηση prompt** | Κρατήστε τις οδηγίες του συστήματος σύντομες. Αφαιρέστε περιττό πλαίσιο για να μειώσετε τα input tokens. |
    "| **Επιλογή μοντέλου** | Χρησιμοποιήστε μικρότερα, φθηνότερα μοντέλα (π.χ. GPT-5-mini) για απλές εργασίες όπως ταξινόμηση ή εξαγωγή και κρατήστε τα μεγαλύτερα μοντέλα για πολύπλοκο συλλογισμό. |\n",
| **Caching** | Αποθηκεύστε αποτελέσματα εργαλείων και συχνές ερωτήσεις για να αποφύγετε περιττές κλήσεις API. |
| **Προϋπολογισμοί tokens** | Ορίστε όρια `max_tokens` για να αποτρέψετε απροσδόκητα μακρές απαντήσεις. |
| **Ομαδοποίηση** | Ομαδοποιήστε πολλαπλές ερωτήσεις χρηστών σε μία κλήση API όπου είναι δυνατόν. |

Στην πράξη, μια προσέγγιση με επίπεδα λειτουργεί καλά: κατευθύνετε απλές αιτήσεις σε ένα γρήγορο, φθηνό μοντέλο και κλιμακώστε μόνο τις σύνθετες ερωτήσεις σε πιο ικανό μοντέλο.


## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να:

1. **Προσθέτετε παρατηρησιμότητα** στις αλληλεπιδράσεις του agent με χρονισμό και καταγραφή, θέτοντας τα θεμέλια για ιχνηλάτηση και παρακολούθηση.
2. **Αξιολογείτε αυτόματα τις απαντήσεις του agent** χρησιμοποιώντας έναν agent αξιολογητή που βαθμολογεί την πληρότητα, την ακρίβεια και τη χρησιμότητα.
3. **Διαχειρίζεστε το κόστος** μέσω βελτιστοποίησης prompt, επιλογής μοντέλου, caching και προϋπολογισμών tokens.

Αυτά τα πρότυπα παραγωγής βοηθούν να διασφαλίσετε ότι οι agents AI σας είναι αξιόπιστοι, μετρήσιμοι και οικονομικοί σε κλίμακα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
